# Week 2 - Agents

### Bu Haftanın Hedefi

Geçen hafta LLM'lerin tool kullanmasını öğrendik.

Bu hafta:

- Agent nedir?
- Tool Calling ile Agent arasındaki fark nedir?
- ReAct (Reasoning + Acting) nedir?
- Agent'lar nasıl karar verir?
- Multi-step workflow nasıl çalışır?
- MCP neden ortaya çıktı?
- Production sistemlerde Agent'lar nasıl kullanılır?

öğreneceğiz.

Bu haftanın sonunda kendi Agent sistemimizi geliştireceğiz.

---

## Öğrenme Hedefleri

Bu notebook sonunda:

✅ Agent kavramını anlayabileceksiniz

✅ ReAct Pattern'i açıklayabileceksiniz

✅ Manual Agent Loop yazabileceksiniz

✅ Birden fazla tool kullanan Agent geliştirebileceksiniz

✅ create_agent() kullanabileceksiniz

✅ MCP'nin neden gerekli olduğunu anlayabileceksiniz

✅ Production Agent mimarisini kavrayabileceksiniz

---


# 1. Agent Nedir?

Birçok kişi Agent kelimesini duyunca:

> "LLM + Tool"

olarak düşünür.

Bu eksik bir tanımdır.

Aslında:

```text
Agent =
LLM
+
Tools
+
Memory
+
State
+
Planning
+
Reasoning
+
Execution Loop
```

Agent'ın en önemli özelliği:

**Bir problemi çözmek için birden fazla adım planlayabilmesidir.**

---

![llm-vs-agent](https://raw.githubusercontent.com/microsoft/langchain-for-beginners/main/05-agents/images/llm-vs-agent.png)

## Klasik LLM

```text
User
 ↓
LLM
 ↓
Answer
```

**Örnek:**

**User:**

"Türkiye'nin başkenti neresidir?"

**LLM:**

"Ankara"



Tek adım.

---

## Tool Calling

```text
User
 ↓
LLM
 ↓
Tool
 ↓
Answer
```


**Örnek:**

**User:**

"İstanbul'da hava nasıl?"

**LLM:**

weather_tool()

**Tool sonucu:**

18°C

**LLM:**

"Hava 18 derece."

---

## Agent

```text
User
 ↓
LLM
 ↓
Tool
 ↓
Observation
 ↓
LLM
 ↓
Another Tool
 ↓
Observation
 ↓
Answer
```

- Birden fazla karar verir.

- Birden fazla tool çağırabilir.

- Gerekirse planını değiştirebilir.

- İşte bu noktada Agent ortaya çıkar.

# 2. Project Manager Analojisi

Agent mantığını anlamanın en kolay yolu budur.

Bir proje yöneticisi düşünün.

Ekibinde:

- Veri Analisti
- Muhasebeci
- Araştırmacı
- Asistan

olsun.

Birisi size şunu soruyor:

> "Bu çeyrek büyüme oranımız nedir?"

Siz doğrudan cevap vermezsiniz.

Süreç:

1. Veri analistine sorarsınız.
2. Sonuçları alırsınız.
3. Muhasebeciye hesaplattırırsınız.
4. Sonucu yorumlarsınız.
5. Cevabı verirsiniz.

Agent'lar da aynı şekilde çalışır.

Tool'lar onların uzman ekip arkadaşlarıdır.

![manager-specialists-analogy](https://github.com/microsoft/langchain-for-beginners/blob/main/05-agents/images/manager-specialists-analogy.png?raw=true)





# 3. ReAct Pattern

ReAct:

Reasoning + Acting

Agent'ların kullandığı temel düşünme modelidir.

---

## Adımlar

1. Thought
2. Action
3. Observation
4. Repeat
5. Final Answer

---

Örnek:

User:

25 * 17 kaçtır ve asal sayı mı?

Thought:

Önce çarpımı hesaplamalıyım.

Action:

calculator()

Observation:

425

Thought:

Şimdi asal mı kontrol etmeliyim.

Action:

is_prime()

Observation:

False

Final Answer:

425 asal değildir.

In [ ]:
def print_react_example():
    print("Thought: Önce çarpımı hesaplamalıyım")
    print("Action: calculator(25*17)")
    print("Observation: 425")

    print()

    print("Thought: Şimdi asal olup olmadığını kontrol etmeliyim")
    print("Action: is_prime(425)")
    print("Observation: False")

    print()

    print("Final Answer: 425 asal değildir.")


print_react_example()

Thought: Önce çarpımı hesaplamalıyım
Action: calculator(25*17)
Observation: 425

Thought: Şimdi asal olup olmadığını kontrol etmeliyim
Action: is_prime(425)
Observation: False

Final Answer: 425 asal değildir.


# 4. İlk Manual Agent Loop

Framework kullanmadan önce Agent'ın altında ne olduğunu görelim.

Amacımız:

Agent'ın sihir olmadığını görmek.

Aslında sadece:

```text
Reason
↓
Tool
↓
Observe
↓
Reason
↓
Tool
↓
Observe
```

döngüsünden oluşur.

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
import json
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key:")
client = OpenAI()


OpenAI API Key:··········


In [ ]:
def company_search(company_name: str):
    companies = {
        "Microsoft": "Microsoft is a global cloud and software company.",
        "Google": "Google is a search and cloud company.",
        "OpenAI": "OpenAI develops AI models and products."
    }

    return companies.get(company_name, "Company not found")

In [ ]:
def recent_news(company_name: str):

    news = {
        "Microsoft": "Microsoft announced new AI features.",
        "Google": "Google expanded Gemini usage.",
        "OpenAI": "OpenAI released a new model."
    }

    return news.get(company_name, "No news found")

# Tool Tanımları

Agent'ın kullanabileceği araçları tanımlıyoruz.

In [ ]:
tools = [
    {
        "type": "function",
        "name": "company_search",
        "description": "Search information about a company.",
        "parameters": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string"
                }
            },
            "required": ["company_name"]
        }
    },
    {
        "type": "function",
        "name": "recent_news",
        "description": "Find recent company news.",
        "parameters": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string"
                }
            },
            "required": ["company_name"]
        }
    }
]

In [ ]:
available_tools = {
    "company_search": company_search,
    "recent_news": recent_news
}

# 5. Manual ReAct Loop

Şimdi Agent'ı kendimiz yazacağız.

Bu bölüm çok önemli.

Çünkü create_agent() gibi frameworklerin arkasında çalışan mantık tam olarak budur.

In [ ]:
def run_agent(query, max_steps=5):

    messages = [
        {
            "role": "user",
            "content": query
        }
    ]

    for step in range(max_steps):

        print(f"\n===== ITERATION {step+1} =====")

        response = client.responses.create(
            model="gpt-4.1-mini",
            input=messages,
            tools=tools
        )

        tool_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        if not tool_calls:
            print("\nFINAL ANSWER:")
            print(response.output_text)
            return

        for call in tool_calls:

            print("\nACTION:")
            print(call.name)

            args = json.loads(call.arguments)

            result = available_tools[call.name](**args)

            print("\nOBSERVATION:")
            print(result)

            messages.append(call)

            messages.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": result
            })

In [ ]:
run_agent(
    "Microsoft hakkında araştırma yap ve son gelişmeleri özetle."
)


===== ITERATION 1 =====

ACTION:
company_search

OBSERVATION:
Microsoft is a global cloud and software company.

ACTION:
recent_news

OBSERVATION:
Microsoft announced new AI features.

===== ITERATION 2 =====

FINAL ANSWER:
Microsoft, global bir yazılım ve bulut hizmetleri şirketidir. Son gelişmeler arasında, Microsoft'un yeni yapay zeka (AI) özelliklerini duyurması öne çıkıyor. Bu özellikler, şirketin teknoloji ve hizmet portföyünü geliştirme odaklı önemli adımlarından biridir. Başka bir konuda detaylı bilgi almak ister misiniz?


# 6. create_agent()

Bir önceki bölümde Agent'ı tamamen kendimiz yazdık.

Bu sayede:

- ReAct Pattern
- Tool Calls
- Observation
- Iteration

gibi kavramların nasıl çalıştığını gördük.

Ancak production sistemlerde her seferinde bu loop'u yazmak istemeyiz.

Bu yüzden Agent Framework'leri ortaya çıkmıştır.

Örnekler:

- LangChain
- LangGraph
- PydanticAI
- OpenAI Agents SDK
- CrewAI
- AutoGen

Bu frameworklerin ortak amacı:

```text
Reason
↓
Tool
↓
Observation
↓
Reason
↓
Tool
↓
Observation
↓
Answer
```

döngüsünü bizim yerimize yönetmektir.

---

Bugün LangChain'in create_agent() yapısını inceleyeceğiz.

Aslında yaptığı şey:

"Manual Agent Loop'u sizin yerinize yazmak"

olarak düşünülebilir.

In [ ]:
!pip install -q langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.3 MB/s eta 0:00:00


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

## Tool Tanımları

LangChain'de tool'lar Python decorator ile tanımlanabilir.

In [ ]:
@tool
def company_search(company_name: str) -> str:
    """
    Search information about a company.
    """

    companies = {
        "Microsoft": "Microsoft is a global cloud and software company.",
        "Google": "Google is a search and cloud company.",
        "OpenAI": "OpenAI develops AI models and products."
    }

    return companies.get(company_name, "Company not found")


@tool
def recent_news(company_name: str) -> str:
    """
    Find recent company news.
    """

    news = {
        "Microsoft": "Microsoft announced new AI features.",
        "Google": "Google expanded Gemini usage.",
        "OpenAI": "OpenAI released a new model."
    }

    return news.get(company_name, "No news found")

In [ ]:
agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[
        company_search,
        recent_news
    ],
    system_prompt="""
      You are a business research assistant.
      Use tools when needed.
    """
)

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Microsoft hakkında araştırma yap. Türkçe dilinde cevapla."
        }
    ]
})

response["messages"][-1].content

'Microsoft, dünya çapında faaliyet gösteren bir bulut ve yazılım şirketidir. Daha detaylı bilgi isterseniz, şirketin faaliyet alanları, tarihçesi, ürünleri veya diğer özel konularda araştırma yapabilirim. Başka hangi bilgilerle ilgileniyorsunuz?'

## Ne Oldu?

Biz artık:

```python
for iteration in range(...):
```

yazmadık.

Ancak Agent yine:

- Tool seçti
- Tool çağırdı
- Sonuçları yorumladı

Çünkü create_agent() bunu arka planda otomatik yapıyor.

Production sistemlerde genellikle bu yaklaşım kullanılır.

# 7. Multi Tool Agent

Agent'ların asıl gücü burada ortaya çıkar.

Birden fazla tool'u aynı anda kullanabilirler.

Bir insan gibi:

- Araştırma yapabilir
- Hesaplama yapabilir
- Sonuçları birleştirebilir

---

Bu noktada Agent'ın davranışı artık önceden yazılmış if/else kurallarıyla değil,

Tool açıklamalarına bakarak verdiği kararlarla belirlenir.

In [ ]:
@tool
def calculator(expression: str) -> str:
    """
    Perform mathematical calculations.
    """

    result = eval(expression)

    return str(result)

In [ ]:
agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[
        company_search,
        recent_news,
        calculator
    ]
)

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
              Microsoft hakkında araştırma yap.
              Sonra 1 milyon doların TL karşılığını
              35 kur üzerinden hesapla.
            """
        }
    ]
})

print(response["messages"][-1].content)

Microsoft, küresel bir bulut ve yazılım şirketidir.

1 milyon doların Türk Lirası karşılığı, 35 kur üzerinden hesaplandığında 35.000.000 TL olur. Başka bir konuda yardımcı olabilir miyim?


## Önemli Nokta

Bu örnekte Agent:

- Hangi tool'u çağıracağını
- Hangi sırayla çağıracağını
- Kaç kez çağıracağını

kendisi belirledi.

İşte Agent sistemlerini klasik workflow'lardan ayıran özellik budur.

# 8. Middleware Nedir?

Production sistemlerde Agent geliştirmek sadece Tool Calling değildir.

Şirketlerin para ödediği kısım genellikle şunlardır:

- Logging
- Monitoring
- Cost Control
- Retry
- Routing
- Security

Bu amaçla Middleware kullanılır.

Middleware:

Agent'ın davranışını değiştiren ara katmandır.

Tıpkı web frameworklerindeki middleware mantığı gibi çalışır.

In [ ]:
from langchain.agents.middleware import AgentMiddleware

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import ToolMessage


class LoggingMiddleware(AgentMiddleware):

    def wrap_tool_call(
        self,
        request,
        handler,
    ):

        tool_call = request.tool_call

        print("\n" + "=" * 60)
        print("TOOL CALL")
        print("=" * 60)

        print(f"Tool Name : {tool_call['name']}")

        if "args" in tool_call:
            print(f"Arguments : {tool_call['args']}")

        result = handler(request)

        print("\nTOOL RESULT")
        print("-" * 60)

        try:
            print(result.content)
        except:
            print(result)

        print("=" * 60 + "\n")

        return result

## Middleware Örnek 1: Logging Middleware

Production sistemlerde:

- Hangi tool kullanıldı?
- Kaç kere kullanıldı?
- Hangi argümanlarla çağrıldı?
- Ne kadar sürdü?

gibi bilgileri takip etmek isteriz.

Bu amaçla middleware kullanılabilir.

In [ ]:
agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[
        company_search,
        recent_news,
        calculator
    ],
    middleware=[
        LoggingMiddleware()
    ]
)

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Microsoft hakkında araştırma yap.
Sonra 1 milyon doların TL karşılığını
35 kur üzerinden hesapla.
"""
        }
    ]
})


TOOL CALL
Tool Name : company_search
Arguments : {'company_name': 'Microsoft'}

TOOL RESULT
------------------------------------------------------------
Microsoft is a global cloud and software company.


TOOL CALL
Tool Name : calculator
Arguments : {'expression': '1000000 * 35'}

TOOL RESULT
------------------------------------------------------------
35000000



LangChain agent çıktısında bütün mesajlar duruyor.

In [ ]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    ToolMessage
)

for idx, message in enumerate(response["messages"]):

    print(f"\nSTEP {idx+1}")
    print("-"*60)

    if isinstance(message, HumanMessage):
        print("USER")
        print(message.content)

    elif isinstance(message, AIMessage):

        if message.tool_calls:

            print("AGENT DECISION")

            for tool_call in message.tool_calls:

                print(
                    f"CALL TOOL -> {tool_call['name']}"
                )

                print(
                    f"ARGS -> {tool_call['args']}"
                )

        else:

            print("AGENT RESPONSE")
            print(message.content)

    elif isinstance(message, ToolMessage):

        print("TOOL RESULT")
        print(message.content)


STEP 1
------------------------------------------------------------
USER

Microsoft hakkında araştırma yap.
Sonra 1 milyon doların TL karşılığını
35 kur üzerinden hesapla.


STEP 2
------------------------------------------------------------
AGENT DECISION
CALL TOOL -> company_search
ARGS -> {'company_name': 'Microsoft'}
CALL TOOL -> calculator
ARGS -> {'expression': '1000000 * 35'}

STEP 3
------------------------------------------------------------
TOOL RESULT
Microsoft is a global cloud and software company.

STEP 4
------------------------------------------------------------
TOOL RESULT
35000000

STEP 5
------------------------------------------------------------
AGENT RESPONSE
Microsoft, küresel çapta bulut ve yazılım alanında faaliyet gösteren bir şirkettir.

Ayrıca, 1 milyon doların 35 TL kurundan hesaplanan Türk Lirası karşılığı 35 milyon TL'dir. Başka bir konuda yardımcı olabilir miyim?


Bu middleware sayesinde:

Her tool çağrısını izleyebiliriz.

Production sistemlerde:

- Audit log
- Debugging
- Monitoring

için çok kullanılır.

## Middleware Örnek 2: Cost Control Middleware

Bu çok gerçek bir senaryo.

Şirketler GPT-5 gibi pahalı modelleri her istekte çalıştırmak istemez.


## Cost Control Middleware

Basit sorularda ucuz model,

karmaşık sorularda güçlü model kullanmak maliyeti ciddi şekilde azaltabilir.

Bu yaklaşım production sistemlerde çok yaygındır.

In [ ]:
from langchain.agents.middleware import AgentMiddleware


class CostControlMiddleware(AgentMiddleware):

    def wrap_model_call(
        self,
        request,
        handler
    ):

        message_count = len(
            request.state["messages"]
        )

        print(
            f"Conversation Length: {message_count}"
        )

        if message_count > 10:
            print(
                "Complex conversation detected"
            )

            print(
                "Switching to stronger model"
            )

        return handler(request)


## Middleware Örnek 3: Tool Error Handling Middleware

Tool'lar her zaman başarılı çalışmayabilir.

Örneğin:

- API timeout
- Rate limit
- Database error
- Network issue

Bu durumda Agent'ın çökmesini istemeyiz.

In [ ]:
from langchain_core.messages import ToolMessage
from langchain.agents.middleware import AgentMiddleware


class ToolErrorMiddleware(
    AgentMiddleware
):

    def wrap_tool_call(
        self,
        request,
        handler
    ):

        try:

            return handler(request)

        except Exception as e:

            tool_name = request.tool_call["name"]

            print(
                f"Tool Error: {tool_name}"
            )

            print(
                f"Reason: {e}"
            )

            return ToolMessage(
                content=f"""
                    Tool failed.

                    Tool Name:
                    {tool_name}

                    Error:
                    {str(e)}
                    """,
                tool_call_id=request.tool_call["id"]
            )

# 9. Agent Architecture

Birçok kişi Agent'ın aşağıdaki gibi çalıştığını düşünür:

```text
User
↓
LLM
↓
Answer
```

Gerçekte production sistemler genellikle şöyledir:

```text
User
↓
Agent
↓
Planner
↓
Tool Router
↓
Tools
↓
Memory
↓
Evaluator
↓
Response
```

Agent tek prompt değildir.

Agent aslında:

- State
- Memory
- Planning
- Routing
- Execution

katmanlarından oluşan bir sistemdir.